# 이전 방식(v5 · v6) vs MILP 비교

MILP로 바꾸기 전의 최적화 방식(원본 `v6`, `v5` 노트북)에 **지금 정한 제약조건을 그대로 주면 어떻게 되는지** 확인하고, 본 모델(MILP)과 같은 표에서 비교하는 노트북이다.

| 항목 | 내용 |
|---|---|
| **입력** | 원본 v6 노트북, `okm_augumented_2021_processed.xlsx`, `Optimizing31_MILP_결과.xlsx` (같은 폴더) |
| **출력** | `이전방식_vs_MILP_비교.xlsx` (5개 시트), `v6_새제약_원본출력.xlsx` (원본 v6 코드가 저장하는 중간 파일) |
| **실행 시간** | 약 20초 (XGBoost 학습 포함, GPU 불필요) |

## 이전 방식이란

| 방식 | 구조 | 인원 |
|---|---|---|
| **v6** (순차 탐색) | 월별로 시간마다 생산량 후보(기존의 50 ~ 150%)를 만들고, XGBoost로 Peak · 전력량을 예측해 비용(전력비 + 생산 편차 + Peak 패널티)이 가장 낮은 후보를 고른다. 월 생산량은 **라그랑주 승수**를 이분 탐색해 기존 100%에 맞춘다. | 기존값 고정 |
| **v5** (인원 유동) | v6 구조에 AI 인원 배분(70 ~ 120%)과 일용직 근무패턴 LP, 폭염 시간대 생산 분산을 더했다. | person-hour 95 ~ 100% |

## 비교하는 다섯 가지

| 번호 | 대상 | 평가 기준 |
|:---:|---|---|
| ① | v6 원본 | XGBoost 예측 (원본 출력값) |
| ② | v5 원본 | XGBoost 예측 (원본 출력값) |
| ③ | **v6 알고리즘 + 우리 제약** (이 노트북에서 실행) | XGBoost 예측 |
| ④ | ③의 생산 계획을 **실측 기준 환산**으로 다시 계산 | 실측 환산 |
| ⑤ | MILP (최종 모델) | 실측 환산 |

> **평가 기준이 다른 이유**: ①②③은 각 방식이 쓰는 XGBoost 예측값으로 잰 값이다. ④⑤는 기존 상태를 실측 그대로 두고 **바뀐 시간만 예측**해서 계산한다. 같은 기준끼리 비교하려면 ③과 ④(이전 방식, 기준만 다름), ④와 ⑤(실측 환산끼리)를 본다.

## 결과 한눈에 보기 (1 ~ 9월 합계, 기존 대비)

| | 전력비 절감 | 인건비 절감 | 총비용 절감 | 연중 최대 Peak (kW) | 2주 생산 미달 |
|---|---:|---:|---:|---|:---:|
| ① v6 원본 | 4,816,815원 (1.28%) | 0 | 4,816,815원 (0.45%) | 243.83 → 243.83 | 9 / 19 |
| ② v5 원본 | 804,318원 (0.21%) | 50,056,673원 (7.19%) | 50,860,991원 (4.74%) | 243.8 → 243.8 | 9 / 19 |
| ③ v6 + 우리 제약 | 4,544,008원 (1.20%) | 0 | 4,544,008원 (0.42%) | 226.7 → 226.3 | 0 / 19 |
| ④ ③을 실측 환산 | −6,874원 (≈ 0%) | 0 | ≈ 0 | 222.0 → 221.9 | 0 / 19 |
| **⑤ MILP** | **67,357,530원 (17.86%)** | **89,299,199원 (12.84%)** | **156,656,729원 (14.60%)** | 222.0 → 222.4 | 0 / 19 |

- 이전 방식은 같은 제약을 줘도 **라인을 끌 수 없고 인원도 고정**이라 줄일 여지가 거의 없다. 총비용 절감액은 MILP가 ③의 **약 34배**이다.
- ③의 전력 절감(1.2%)은 XGBoost가 "생산을 줄이면 전력이 준다"고 반응한 결과다. 같은 계획을 실측 기준으로 다시 계산하면(④) 변화가 사실상 0이다. 가동 중에는 생산량이 전력에 거의 영향을 주지 않기 때문이다.
- 7월 저녁(18 ~ 23시) 생산 비중은 이전 방식이 25.1%에서 33.3%로 **늘렸고**, MILP는 14.9%로 **줄였다.** 이전 방식의 목적함수에는 노동 비용과 1인당 생산량이 없다.

## 이 노트북이 하는 일

| 순서 | 내용 |
|:---:|---|
| 1 | **원본 v6 노트북 코드를 그대로 실행**하되 우리 제약만 추가한다. |
| 2 | 그 결과 계획을 **실측 기준 환산**으로 다시 평가하고, 인원 고정 계획이 요구하는 1인당 생산성과 라인 규칙 충족 여부를 점검한다. |
| 3 | 원본 v6 · v5 결과, 이전 방식 + 우리 제약, MILP 결과를 한 표로 저장한다. |

## 준비물 (같은 폴더)

| 파일 | 설명 |
|---|---|
| `Optimizing15_production_timing_v6.ipynb` | 원본 v6 노트북 (`V6_NOTEBOOK_PATH`). 이 노트북이 코드 셀을 읽어 실행한다. |
| `okm_augumented_2021_processed.xlsx` | 전처리를 마친 데이터. 원본 v6 코드가 읽는 파일명과 같아야 한다. |
| `Optimizing31_MILP_결과.xlsx` | 최적화 노트북 `Optimizing31_MILP.ipynb`의 결과 (`MILP_RESULT_PATH`) |

필요한 라이브러리: `numpy`, `pandas`, `scipy`, `scikit-learn`, `xgboost`, `seaborn`, `matplotlib`, `openpyxl`

> **주의**: 우리 제약은 원본 v6 코드(28번 셀의 `run_sequential_v5`)를 문자열로 치환해 끼워 넣는다. 원본 코드가 바뀌면 `assert`에서 멈추도록 해 두었다.

---

## 1. 설정 · 원본 결과

경로와 우리 제약, 원본 노트북이 낸 결과를 정의한다.

| 설정 | 내용 |
|---|---|
| `BOOST_RULES` | 월 · 시간대별 시간당 생산 상한. 3 · 5월 낮(09 ~ 17시) 175%, 6 · 8월 낮 230%, 7월 낮 300%, 7월 07 ~ 08시 · 저녁(18 ~ 23시) 200%. 기본 상한은 150%이다. MILP와 같은 값이다. |
| `BOOST_CAP_QUANTILE` | 올린 상한도 기존 시간당 생산의 99분위(약 4,727개)를 넘지 않게 하는 설비 안전장치 |
| `CLEAN_MONTHS` | 생산 1개당 전력 영향을 추정하는 정상월 (4, 5, 7, 8, 9월) |
| `ORIG` | 원본 v5 · v6 노트북이 출력한 1 ~ 9월 누계 (전력비 · 인건비 절감, 최대 Peak, 2주 미달 구간). 직접 옮겨 둔 상수이다. |

In [1]:
# ============================================================
# 이전 방식(v6 순차 탐색 + 라그랑주 승수) vs MILP 비교
#  - 이전 방식 = 원본 v6 노트북을 그대로 실행하되, 우리가 정한 제한만 추가해서 돌린다
#  - 비교 대상 MILP 결과 = Optimizing31_MILP_결과.xlsx
# ============================================================
import json, io, sys, contextlib, warnings, numpy as np, pandas as pd
from scipy.optimize import lsq_linear
import matplotlib; matplotlib.use("Agg")
warnings.filterwarnings("ignore")
V6_NOTEBOOK_PATH = "Optimizing15_production_timing_v6.ipynb"      # 원본 v6 노트북
DATA_PATH        = "okm_augumented_2021_processed.xlsx"           # v6 노트북이 읽는 데이터 (현재 폴더)
MILP_RESULT_PATH = "Optimizing31_MILP_결과.xlsx"                    # 최종 MILP 결과
OUT_PATH         = "이전방식_vs_MILP_비교.xlsx"
# 우리가 정한 제한 (MILP와 동일)
BOOST_RULES = [((3, 5), (9, 17), 1.75), ((6, 8), (9, 17), 2.3), ((7,), (9, 17), 3.0), ((7,), (7, 8), 2.0), ((7,), (18, 23), 2.0)]   # (월, (시작시, 끝시), 시간당 생산 상한 배율)
BOOST_CAP_QUANTILE = 0.99      # 올린 상한도 기존 시간당 생산 99분위를 넘지 않게 (설비 안전장치)
CLEAN_MONTHS = [4, 5, 7, 8, 9]
# 원본 노트북이 낸 결과 (원본 출력값; 각 노트북의 1~9월 누계)
ORIG = {"v6": dict(전력비절감_원=4816815, 기존전력비_원=377527585, 인건비변화_원=0, 기존인건비_원=695732321, 최대Peak=("243.83", "243.83"), 미달구간="9 / 19 (최소 86%)", 설명="원본 v6: 인원 고정, 생산 시점만 최적화 (월 100%, 시간별 50~150%)"),
        "v5": dict(전력비절감_원=804318, 기존전력비_원=377527585, 인건비변화_원=-50056673, 기존인건비_원=695732321, 최대Peak=("243.8", "243.8"), 미달구간="9 / 19 (최소 91.8%)", 설명="원본 v5: 인원 유동(person-hour -5%) + 일용직 근무패턴 LP + 폭염 분산")}

---

## 2. 이전 방식 실행 (원본 v6 + 우리 제약)

원본 v6 노트북의 코드 셀을 순서대로 실행한다. 다음 세 가지만 추가한다.

| 추가한 제약 | 내용 |
|---|---|
| 월 · 시간대별 생산 상한 | `BOOST_RULES`를 후보 생산량의 상한에 적용 |
| 최적화 대상 근무일 | 평일에 시작하고 24시간이 온전한 생산일만 결정하고, 주말 · 휴무일은 기존 값으로 고정 |
| 월 × 2주 생산 100% | 월과 2주 구간의 교집합마다 라그랑주 승수를 따로 찾아 **월 · 2주 목표를 모두** 만족 |

실행 후 XGBoost 예측 기준의 전력량 · 전기요금 · Peak와 생산 유지율을 출력한다. 인원은 고정이므로 인건비 변화는 없다.

In [2]:
# ============================================================
# 이전 방식 실행: 원본 v6 코드 + 우리 제한
#  추가한 제한: ① 월·시간대별 생산 상한(상향 규칙+안전장치) ② 최적화 대상은 평일 시작 생산일(주말·휴무는 기존 유지)
#               ③ 생산 100% 이상을 (월 x 2주) 구간별로 요구 -> 월·2주 목표 모두 충족
# ============================================================
nb = json.load(open(V6_NOTEBOOK_PATH)); cells = {i: ''.join(c['source']) for i, c in enumerate(nb['cells']) if c['cell_type'] == 'code'}
EXTRA = """
BOOST_RULES = %r
BOOST_CAP_QUANTILE = %r
def ratio_hi_array(months_, hours_):
    r = np.full(len(months_), float(MAX_PRODUCTION_RATIO)); b = np.zeros(len(months_), bool)
    for ms, (h0, h1), rr in BOOST_RULES:
        sel = np.isin(months_, ms) & (hours_ >= h0) & (hours_ <= h1); r[sel] = np.maximum(r[sel], rr); b |= sel
    return r, b
""" % (BOOST_RULES, BOOST_CAP_QUANTILE)
c28 = cells[28].replace('def run_sequential_v5(data, weekly_limits, max_rows=None, desc="Optimizing15 v6"):', EXTRA + '\ndef run_sequential_v5(data, weekly_limits, max_rows=None, desc="Optimizing15 v6"):')
c28 = c28.replace('    fixed = (p0 <= 0) | (n0 <= 0)\n', '''    fixed = (p0 <= 0) | (n0 <= 0)
    _sd = (dt - pd.Timedelta(hours=8)).dt.normalize(); _g = pd.DataFrame({"sd": _sd, "p": p0})
    _prod = _g.groupby("sd")["p"].transform("sum").values; _len = _g.groupby("sd")["p"].transform("size").values
    _dec = (_sd.dt.dayofweek.values < 5) & (_len == 24) & (_prod > 0); fixed = fixed | (~_dec)
    win_ = ((dt - dt.iloc[0]).dt.days // 14).values; grp = months * 100 + win_
    _rhi, _boost = ratio_hi_array(months, hours); _cap_obs = float(np.quantile(p0[p0 > 0], BOOST_CAP_QUANTILE))
''')
c28 = c28.replace('        for m in np.unique(months[op]):\n            idx = op[months[op]==m]', '        for m in np.unique(grp[op]):\n            idx = op[grp[op]==m]')
c28 = c28.replace('            p_hi = np.minimum(p0[idx]*MAX_PRODUCTION_RATIO,p_cap)', '            p_hi = np.minimum(np.maximum(p0[idx]*MAX_PRODUCTION_RATIO, np.where(_boost[idx], np.minimum(_rhi[idx]*p0[idx], _cap_obs), 0.0)), p_cap)')
c28 = c28.replace('            R = MONTHLY_PRODUCTION_RATIO*p0[months==m].sum() - p0[(months==m)&fixed].sum()', '            R = MONTHLY_PRODUCTION_RATIO*p0[grp==m].sum() - p0[(grp==m)&fixed].sum()')
assert all(k in c28 for k in ['grp[op]==m', '_boost[idx]', 'grp==m', '~_dec']), "원본 v6 코드와 패치 위치가 달라졌습니다"
cells[28] = c28; cells[33] = cells[33].replace("OUTPUT_FILE = 'Optimizing15_AI_생산시점_Peak최적화_v6.xlsx'", "OUTPUT_FILE = 'v6_새제약_원본출력.xlsx'")
LG = {'display': print}
with contextlib.redirect_stdout(io.StringIO()):
    for i in sorted(cells):
        if i in (35, 36, 37, 38, 39): continue
        exec(cells[i], LG)
leg_df, leg_kpi, leg_bi = LG['result_df'].copy(), LG['monthly_kpi'].copy(), LG['BIWEEK_CHECK'].copy(); leg_df['datetime'] = pd.to_datetime(leg_df['datetime'])
L_E0, L_E1 = leg_kpi['기존전력량kWh'].sum(), leg_kpi['최적전력량kWh'].sum(); L_C0, L_C1 = leg_kpi['기존전기요금'].sum(), leg_kpi['최적전기요금'].sum()
print(f"[이전 방식 + 우리 제한, 자체 기준(XGBoost 예측)] 전력량 {L_E0:,.0f} -> {L_E1:,.0f} kWh ({(L_E1/L_E0-1)*100:+.2f}%) | 전기요금 {L_C0:,.0f} -> {L_C1:,.0f}원 (절감 {L_C0-L_C1:,.0f}원) | 연중 최대 안전Peak {leg_kpi['기존최대Peak'].max():.2f} -> {leg_kpi['최적최대Peak'].max():.2f}")
print(f"  2주 생산 100% 미달 구간 {(leg_bi['유지율(%)'] < 100 - 1e-6).sum()} / {len(leg_bi)} | 월 최소 유지율 {leg_kpi['생산량유지율(%)'].min():.2f}% | 인원 변화 0 (고정)")

[이전 방식 + 우리 제한, 자체 기준(XGBoost 예측)] 전력량 2,302,331 -> 2,275,310 kWh (-1.17%) | 전기요금 377,317,522 -> 372,773,514원 (절감 4,544,008원) | 연중 최대 안전Peak 226.70 -> 226.27
  2주 생산 100% 미달 구간 0 / 19 | 월 최소 유지율 100.08% | 인원 변화 0 (고정)


---

## 3. 우리 기준 재평가 · 점검

③의 생산 계획을 실측 기준으로 다시 계산(④)하고, 이전 방식이 우리 제약을 지키는지 점검한다.

| 점검 | 내용 |
|---|---|
| 실측 환산 | 기존 상태는 실측 그대로 두고, 생산량이 바뀐 만큼만 가동 중 생산 1개당 Peak · 전력량 증가분(정상월에서 추정)을 더해 전력비 · Peak를 다시 계산 |
| 1인당 생산성 | 인원이 고정된 계획이 요구하는 1인당 생산량이 시간대별 기존 95분위를 넘는 비율 (기존은 정의상 5%) |
| 라인 규칙 | 라인 종료 시간, 생산일 안에서 쉬는 시간이 그대로 남는지 |
| 7월 시간대 비중 | 기존 · 이전 방식 · MILP의 시간대별(00 ~ 08 / 09 ~ 17 / 18 ~ 23시) 생산 비중 |

In [3]:
# ============================================================
# 우리 기준(실측 기준 환산)으로 이전 방식의 plan 재평가 + 생산성·라인 규칙 점검
# ============================================================
d0 = pd.read_excel(DATA_PATH); d0["datetime"] = pd.to_datetime(d0["datetime"]); d0 = d0.sort_values("datetime").reset_index(drop=True)
d0["hour"], d0["month"], d0["dow"] = d0["datetime"].dt.hour, d0["datetime"].dt.month, d0["datetime"].dt.dayofweek
p0 = d0["생산량"].astype(float).values; n0 = (d0["공장인원"] * 10).values; P0 = d0["전력 최대값"].astype(float).values; E0 = d0["전력 구간합계"].astype(float).values
tariff = d0["전기요금(계절)"].astype(float).values; hour = d0["hour"].values; mo = d0["month"].values
_sd = (d0["datetime"] - pd.Timedelta(hours=8)).dt.normalize(); _g = pd.DataFrame({"sd": _sd, "p": p0}); _prod = _g.groupby("sd")["p"].transform("sum").values; _len = _g.groupby("sd")["p"].transform("size").values
dec = (_sd.dt.dayofweek.values < 5) & (_len == 24) & (_prod > 0)
# 가동 중 생산 1개당 Peak·전력량 증가 (정상월, 시간대·날씨 통제, 계수 >= 0)
def fit_slope(target):
    m = d0["month"].isin(CLEAN_MONTHS).values & dec & (p0 > 0); c = d0[m]
    H = pd.get_dummies(pd.Categorical(c["hour"], categories=range(24))).values.astype(float); base = np.column_stack([H, c["기온"], c["기온"] ** 2, c["습도"], c["풍속"]]); X = np.column_stack([base, c["생산량"] / 100]); k = base.shape[1]
    return lsq_linear(X, c[target].values, bounds=(np.r_[np.full(k, -np.inf), 0], np.full(X.shape[1], np.inf))).x[k] / 100
bq, gq = fit_slope("전력 최대값"), fit_slope("전력 구간합계")
q = d0[["datetime"]].merge(leg_df[["datetime", "최적생산량"]], on="datetime", how="left")["최적생산량"].values.astype(float)
on_ = (p0 > 0)
En, Pn = E0 + gq * (q - p0) * on_, P0 + bq * (q - p0) * on_
mm = pd.Series(mo); months_ = sorted(set(mo))
OUR = pd.DataFrame({"기존 전력비(원)": pd.Series(E0 * tariff).groupby(mm).sum().values, "이전방식 전력비(원)": pd.Series(En * tariff).groupby(mm).sum().values, "기존 최대Peak": pd.Series(P0).groupby(mm).max().values, "이전방식 최대Peak": pd.Series(Pn).groupby(mm).max().values}, index=months_)
OUR["전력비 절감(원)"] = OUR["기존 전력비(원)"] - OUR["이전방식 전력비(원)"]
# 인원 고정 plan 이 요구하는 1인당 생산성: 시간대별 기존 1인당 생산량 95분위 초과 여부
on2 = (p0 > 0) & (n0 > 0); rho0 = np.where(on2, p0 / np.maximum(n0, 1e-9), np.nan); hi_h = pd.Series(rho0[on2]).groupby(hour[on2]).quantile(0.95).reindex(range(24)).values
ratio = q / np.maximum(n0, 1e-9) / hi_h[hour]; band_viol = int((on2 & (ratio > 1)).sum()); band_pct = band_viol / int(on2.sum()) * 100; band_max = float(np.nanmax(np.where(on2, ratio, np.nan)))
idle_hours = int((dec & (p0 <= 0)).sum()); stop_hours = int((dec & (p0 > 0) & (q <= 0)).sum())
blk = pd.cut(hour, [-1, 8, 17, 23], labels=["00-08시", "09-17시", "18-23시"]); sel7 = dec & (mo == 7)
SH7 = pd.DataFrame({"기존": pd.Series(p0[sel7]).groupby(blk[sel7]).sum(), "이전 방식+새 제한": pd.Series(q[sel7]).groupby(blk[sel7]).sum()}); SH7 = (SH7 / SH7.sum() * 100).round(1)
S = pd.read_excel(MILP_RESULT_PATH, sheet_name="전체_최적스케줄"); S = S[(S["월"] == 7) & (S["최적화대상"] == 1)].copy(); S["블록"] = pd.cut(S["시"], [-1, 8, 17, 23], labels=["00-08시", "09-17시", "18-23시"])
_m = S.groupby("블록", observed=True)["최적생산량"].sum(); SH7["MILP"] = (_m / _m.sum() * 100).round(1).values
print(f"[우리 기준 재평가] 전력량 {(En.sum()/E0.sum()-1)*100:+.4f}% | 전력비 절감 {OUR['전력비 절감(원)'].sum():,.0f}원 | 연중 최대 Peak {P0.max():.1f} -> {Pn.max():.1f}")
print(f"[인원 고정 plan 점검] 기존 1인당 생산성 95분위 초과 {band_viol:,}/{int(on2.sum()):,}시간 ({band_pct:.1f}%, 기존은 5%) | 최대 {band_max:.2f}배 | 라인 종료 {stop_hours}시간 | 생산일 안 쉬는 시간 {idle_hours}시간 유지")
print("7월 생산 시간대 비중(%):"); print(SH7.to_string())

[우리 기준 재평가] 전력량 +0.0017% | 전력비 절감 -6,874원 | 연중 최대 Peak 222.0 -> 221.9
[인원 고정 plan 점검] 기존 1인당 생산성 95분위 초과 500/3,511시간 (14.2%, 기존은 5%) | 최대 2.10배 | 라인 종료 0시간 | 생산일 안 쉬는 시간 573시간 유지
7월 생산 시간대 비중(%):
          기존  이전 방식+새 제한  MILP
00-08시  16.3        11.5   4.6
09-17시  58.7        55.1  80.5
18-23시  25.1        33.3  14.9


---

## 4. 비교표 저장

`이전방식_vs_MILP_비교.xlsx`에 다음 시트를 저장하고 요약표를 출력한다.

| 시트 | 내용 |
|---|---|
| 요약 비교 | 다섯 방법의 전력비 · 인건비 · 총비용 절감, 최대 Peak, 2주 미달 구간 |
| 제약 충족 비교 | 월 · 2주 생산, 시간별 상 · 하한, 안전마진, 라인 종료, 인원 배치, 1인당 생산량, 전력 인과 반영 여부 |
| 월별 비교 | 월별 전력비 절감 (③ 자체 기준, ④ 실측 환산, ⑤ MILP)과 MILP의 인건비 · 총비용 |
| 7월 생산 시간대(%) | 기존 · 이전 방식 · MILP의 7월 시간대별 생산 비중 |
| 방법 설명 | 이전 방식과 MILP의 구조, 한계, 비교 기준 주의 |

> **해석 시 유의**: ③은 같은 제약을 줬을 때 이전 방식이 낼 수 있는 값이고, 결정할 수 없는 항목(라인 종료, 정수 근무 배치, 1인당 생산량 용량)은 **적용 불가**로 표시된다. 위 요약표(결과 한눈에 보기)의 숫자는 이 노트북의 실행 결과와 같다.

In [4]:
# ============================================================
# 비교표 저장: 요약 / 제약 충족 / 월별 / 7월 시간대 / 설명
# ============================================================
M = pd.read_excel(MILP_RESULT_PATH, sheet_name="월별_리포트(전체항목)"); M["월"] = M["월"].astype(str); Mt = M[M["월"] == "합계"].iloc[0]; Mm = M[M["월"] != "합계"].copy(); Mm["월"] = Mm["월"].astype(int)
def row(name, desc, e_save, e0, lab_chg, lab0, peak, miss, basis):
    c_save = e_save - lab_chg; c0 = e0 + lab0
    return {"방법": name, "설명": desc, "전력비 절감(만원)": round(e_save / 1e4), "전력비 변화(%)": round(-e_save / e0 * 100, 2), "인건비 변화(만원)": round(lab_chg / 1e4), "인건비 변화(%)": round(lab_chg / lab0 * 100, 2),
            "총비용 절감(만원)": round(c_save / 1e4), "총비용 변화(%)": round(-c_save / c0 * 100, 2), "연중 최대 Peak(기존→최적, kW)": peak, "2주 생산 미달 구간": miss, "평가 기준": basis}
o6, o5 = ORIG["v6"], ORIG["v5"]; e0 = o6["기존전력비_원"]; lab0 = o6["기존인건비_원"]
SUMMARY = pd.DataFrame([
    row("① 이전 방식 v6 (원본)", o6["설명"], o6["전력비절감_원"], e0, 0, lab0, f"{o6['최대Peak'][0]} → {o6['최대Peak'][1]}", o6["미달구간"], "XGBoost 예측(자체)"),
    row("② 이전 방식 v5 (원본)", o5["설명"], o5["전력비절감_원"], e0, o5["인건비변화_원"], lab0, f"{o5['최대Peak'][0]} → {o5['최대Peak'][1]}", o5["미달구간"], "XGBoost 예측(자체)"),
    row("③ 이전 방식 v6 알고리즘 + 우리 제한", "원본 v6를 그대로 실행 + 월·시간대별 생산 상한, 월×2주 생산 100%, 최적화 대상 근무일 (인원 고정)", L_C0 - L_C1, L_C0, 0, lab0, f"{leg_kpi['기존최대Peak'].max():.1f} → {leg_kpi['최적최대Peak'].max():.1f}", f"{(leg_bi['유지율(%)'] < 100 - 1e-6).sum()} / {len(leg_bi)}", "XGBoost 예측(자체)"),
    row("④ ③을 우리 기준으로 재평가", "③의 생산 plan을 실측 기준 환산(인과 모델)으로 다시 계산", OUR["전력비 절감(원)"].sum(), OUR["기존 전력비(원)"].sum(), 0, lab0, f"{P0.max():.1f} → {Pn.max():.1f}", f"{(leg_bi['유지율(%)'] < 100 - 1e-6).sum()} / {len(leg_bi)}", "실측 기준 환산"),
    row("⑤ MILP (최종)", "라인 종료·정수 근무패턴·휴게 규칙·단기 근무자 50% 미만까지 함께 결정", Mt["전력비 절감(원)"], Mt["기존 전력비(원)"], Mt["인건비 변화(원)"], Mt["기존 인건비(원)"], f"{Mt['기존 최대Peak(실측,kW)']:.1f} → {Mt['최적 최대Peak(환산,kW)']:.1f}", "0 / 19", "실측 기준 환산")])
CONS = pd.DataFrame([
    ["월 생산 100% 이상", "O (원본 v6의 유일한 hard 제약)", "O"], ["2주 생산 100% 이상", f"△ 원본은 진단만(9/19 미달). 월×2주 구간 승수로 확장하면 충족({(leg_bi['유지율(%)'] < 100 - 1e-6).sum()}/{len(leg_bi)} 미달)", "O"],
    ["시간별 생산 상·하한 (월·시간대별 상향 포함)", "O (후보 격자의 범위로 적용)", "O"], ["안전마진(과소예측 방지)", "O (구간별 마진, XGBoost 예측에 가산)", "O (시간×달 차등 마진)"],
    ["08시 가동 시작 · 한 번 끄면 연속 정지 · 라인 종료", f"X 라인을 끌 수 없음 (정지 {stop_hours}시간, 생산일 안 쉬는 시간 {idle_hours}시간 그대로)", "O"],
    ["인원·근무 배치 (정수 인원, 휴게, 단기 근무자 50% 미만)", "X 인원 고정 (v5는 인원 유동이나 근무규칙은 사후 LP)", "O"],
    ["1인당 생산량(ML) 용량 이내", f"X 고려하지 않음: plan이 요구하는 1인당 생산성이 기존 95분위 초과 {band_pct:.1f}%(기존 5%), 최대 {band_max:.2f}배", "O"],
    ["생산이 전력에 미치는 영향(인과)", f"X XGBoost의 lag 반응에 의존: 자체 기준 전력량 {(L_E1/L_E0-1)*100:+.2f}% → 실측 기준 환산 {(En.sum()/E0.sum()-1)*100:+.3f}%", "O (가동 상태·생산량을 분리)"]], columns=["제한/기능", "이전 방식(v6 알고리즘)", "MILP"])
MON = pd.DataFrame({"월": months_, "③ 전력비 절감(자체 기준, 만원)": (leg_kpi["전기요금절감액"].values / 1e4).round(0), "④ 전력비 절감(실측 기준 환산, 만원)": (OUR["전력비 절감(원)"].values / 1e4).round(1),
                    "⑤ MILP 전력비 절감(만원)": (Mm["전력비 절감(원)"].values / 1e4).round(0), "⑤ MILP 인건비 변화(만원)": (Mm["인건비 변화(원)"].values / 1e4).round(0), "⑤ MILP 총비용 절감(만원)": (Mm["총비용 절감(원)"].values / 1e4).round(0), "⑤ MILP 총비용 변화(%)": (-Mm["총비용 절감(%)"].values).round(1)})
EXPL = pd.DataFrame({"구분": ["이전 방식 (v6)", "이전 방식의 한계", "MILP (현재)", "비교 기준 주의"], "설명": [
    "월별로 시간마다 생산량 후보(기존의 50~150%)를 만들고 XGBoost로 Peak·전력량을 예측해 비용(전력비 + 생산 편차 + Peak 패널티)이 최소인 후보를 고른 뒤, 월 생산량을 기존 100%에 맞추는 라그랑주 승수를 찾는 순차 탐색. 인원은 기존값 고정.",
    "라인을 끄거나 인원·근무형태를 정하지 못하고, 노동·1인당 생산량을 목적에 넣지 않아 인건비 절감이 불가능. 전력 절감은 XGBoost가 생산량 변화에 반응하는 것에서 나오며 실측 기준으로는 거의 0.",
    "근무일별 라인 가동/종료(이진변수) + 시간별 생산 + 일용직 근무패턴(정수 인원, 휴게, 단기 근무자 비율)을 함께 결정. 가동 상태·생산량을 분리한 전력 모델과 시간×달 차등 안전마진 사용.",
    "①②③은 각 방식의 XGBoost 예측 기준, ④⑤는 실측 기준 환산(기존 상태는 실측 그대로, 바뀐 시간만 예측). 인건비 기준은 기존 기록(소수 인원×시급) 6.96억원."]})
with pd.ExcelWriter(OUT_PATH, engine="openpyxl") as w:
    SUMMARY.to_excel(w, sheet_name="요약 비교", index=False); CONS.to_excel(w, sheet_name="제약 충족 비교", index=False); MON.to_excel(w, sheet_name="월별 비교", index=False)
    SH7.reset_index().rename(columns={"index": "시간대", "블록": "시간대"}).to_excel(w, sheet_name="7월 생산 시간대(%)", index=False); EXPL.to_excel(w, sheet_name="방법 설명", index=False)
    for ws in w.book.worksheets:
        ws.freeze_panes = "A2"
        for col in ws.columns: ws.column_dimensions[col[0].column_letter].width = min(max(10, max(len(str(c.value)) if c.value is not None else 0 for c in col[:60]) * 1.7 + 2), 70)
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30); pd.set_option("display.max_colwidth", 60)
print("저장 완료:", OUT_PATH); print(SUMMARY.drop(columns=["설명"]).to_string(index=False))

저장 완료: 이전방식_vs_MILP_비교.xlsx
                     방법  전력비 절감(만원)  전력비 변화(%)  인건비 변화(만원)  인건비 변화(%)  총비용 절감(만원)  총비용 변화(%) 연중 최대 Peak(기존→최적, kW)       2주 생산 미달 구간          평가 기준
        ① 이전 방식 v6 (원본)         482      -1.28           0       0.00         482      -0.45       243.83 → 243.83   9 / 19 (최소 86%) XGBoost 예측(자체)
        ② 이전 방식 v5 (원본)          80      -0.21       -5006      -7.19        5086      -4.74         243.8 → 243.8 9 / 19 (최소 91.8%) XGBoost 예측(자체)
③ 이전 방식 v6 알고리즘 + 우리 제한         454      -1.20           0       0.00         454      -0.42         226.7 → 226.3            0 / 19 XGBoost 예측(자체)
       ④ ③을 우리 기준으로 재평가          -1       0.00           0       0.00          -1       0.00         222.0 → 221.9            0 / 19       실측 기준 환산
            ⑤ MILP (최종)        6736     -17.86       -8930     -12.84       15666     -14.60         222.0 → 222.4            0 / 19       실측 기준 환산
